# Week 6 Exercise: Price Estimator Baseline + Error Analysis

This notebook builds a **baseline price predictor** from item descriptions using TF‑IDF + Ridge regression.
It evaluates MAE/RMSE, compares against a median baseline, and performs simple error analysis.


In [ ]:
# If needed, install dependencies
# !pip -q install scikit-learn numpy


In [1]:
# Imports
import csv
import math
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
# Load data
DATA_PATH = Path('/Users/andela/projects/llm_engineering/week6/human_out.csv')

def load_csv(path: Path):
    texts = []
    prices = []
    with path.open() as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 2:
                continue
            text = row[0].strip()
            try:
                price = float(row[1])
            except ValueError:
                continue
            texts.append(text)
            prices.append(price)
    return texts, prices

texts, prices = load_csv(DATA_PATH)
print('Loaded:', len(texts))
if len(texts) == 0:
    raise ValueError('No data loaded. Check DATA_PATH.')


Loaded: 100


In [3]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    texts, prices, test_size=0.2, random_state=42
)


In [4]:
# Baseline: predict the median price
median_price = float(np.median(y_train))
baseline_preds = [median_price] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = math.sqrt(mean_squared_error(y_test, baseline_preds))
print('Baseline median price:', round(median_price, 2))
print('Baseline MAE:', round(baseline_mae, 2))
print('Baseline RMSE:', round(baseline_rmse, 2))


Baseline median price: 50.0
Baseline MAE: 49.7
Baseline RMSE: 81.66


In [5]:
# Model: TF-IDF + Ridge on log1p(price)
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)
regressor = Ridge(alpha=1.0)
pipeline = Pipeline([
    ('tfidf', tfidf),
    ('ridge', regressor)
])
model = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)
model.fit(X_train, y_train)


,regressor,"Pipeline(step...e', Ridge())])"
,transformer,None
,func,<ufunc 'log1p'>
,inverse_func,<ufunc 'expm1'>
,check_inverse,True
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None


In [6]:
# Evaluate
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = math.sqrt(mean_squared_error(y_test, preds))
print('Model MAE:', round(mae, 2))
print('Model RMSE:', round(rmse, 2))


Model MAE: 44.12
Model RMSE: 71.64


In [7]:
# Error analysis: show top 5 worst predictions
def shorten(text, n=180):
    return text[:n] + ('...' if len(text) > n else '')

errors = []
for text, actual, pred in zip(X_test, y_test, preds):
    errors.append((abs(actual - pred), actual, pred, text))
errors.sort(reverse=True)

for i in range(5):
    err, actual, pred, text = errors[i]
    print('\n---')
    print('Actual:', round(actual, 2), 'Pred:', round(pred, 2), 'AbsErr:', round(err, 2))
    print(shorten(text))



---
Actual: 350.0 Pred: 87.95 AbsErr: 262.05
Title: Dell Optiplex 390 19" Desktop Bundle  
Category: Computer & Peripherals  
Brand: Dell  
Description: All-in-one Dell Optiplex 390 desktop with i3 processor, 4 GB RAM, 1 TB H...

---
Actual: 185.0 Pred: 76.74 AbsErr: 108.26
Title: APS iBoard 6‑inch Aluminum Running Boards  
Category: Automotive Accessories  
Brand: APS  
Description: Heavy‑duty 6‑inch aluminum running boards for Nissan Frontier Crew C...

---
Actual: 120.0 Pred: 39.97 AbsErr: 80.03
Title: Pidoko Kids Train Table – Grey 90‑Piece Train Set  
Category: Toys & Games  
Brand: Pidoko Kids  
Description: A sturdy, large wooden train table with a 90‑piece train set, ...

---
Actual: 120.0 Pred: 53.86 AbsErr: 66.14
Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay...

---
Actual: 110.0 Pred: 50.35 AbsErr: 59.65
Title: HK 13" Heavy Duty Black Push-O

In [8]:
# Quick prediction helper
def predict_price(description: str) -> float:
    return float(model.predict([description])[0])

sample = ('Title: Example Headphones\n'
          'Category: Electronics\n'
          'Brand: ExampleCo\n'
          'Description: Wireless over-ear headphones with noise cancellation and 30-hour battery life.\n'
          'Details: Includes carrying case, fast USB-C charging, and 2-year warranty.')
print('Predicted price:', round(predict_price(sample), 2))


Predicted price: 48.9
